<a href="https://colab.research.google.com/github/khu3086/FastAICourse/blob/main/Lec02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Project: Interactive Bear Classifier
This notebook demonstrates a complete machine learning pipeline using `fastai`. Due to search API limitations, we use a sample dataset to simulate the classification of Grizzly, Black, and Teddy bears.

### Note on GitHub Rendering
If you see a `the 'state' key is missing from 'metadata.widgets'` error on GitHub, it is because of how interactive widgets are saved. You can fix this by going to **Edit -> Notebook settings** and toggling **'Include widget state in further saves of this notebook'**.

In [129]:
# This cell ensures that we don't save broken widget state in the metadata if you are pushing to Git
import json
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

## 1. Setup and Dependencies
We begin by installing the `fastbook` library and importing the necessary vision and widget modules from `fastai`.

In [124]:
import sys
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

if 'google.colab' in sys.modules:
    !pip install -Uqq fastbook

from fastai.vision.all import *
from fastai.vision.widgets import *
from fastbook import *
import os, shutil
from pathlib import Path

## 2. Data Gathering (Simulation)
To avoid API rate-limiting issues, we utilize the Oxford-IIIT Pet Dataset as a reliable source to simulate our 'Grizzly', 'Black', and 'Teddy' bear categories.

In [125]:
path = Path('bears')
if path.exists(): shutil.rmtree(path)

print("Downloading and preparing dataset...")
path_data = untar_data(URLs.PETS)
path_data_ims = path_data/'images'
path.mkdir(exist_ok=True)

for bear_type in ['grizzly', 'black', 'teddy']:
    dest = path/bear_type
    dest.mkdir(exist_ok=True)
    sample_ims = get_image_files(path_data_ims)
    for i in range(min(20, len(sample_ims))):
        shutil.copy(sample_ims[i], dest/f'{bear_type}_{i}.jpg')

fns = get_image_files(path)
print(f"Data Prep Complete: {len(fns)} images found.")

Data Prep Complete: 60 images found.


## 3. Data Pipeline
We use the `DataBlock` API to define the transformation pipeline: loading images, assigning labels from folder names, splitting the data, and resizing images to 128x128.

In [126]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)
)
dls = bears.dataloaders(path)

## 4. Model Training
We use Transfer Learning with a ResNet18 architecture. `fine_tune` will train the head of the model and then the entire model for a few epochs.

In [127]:
print("Fine-tuning ResNet18...")
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(2)
learn.export()
print("Model trained and exported as export.pkl")

Fine-tuning ResNet18...


epoch,train_loss,valid_loss,error_rate,time
0,nan,1.617345,0.500000,00:01


/usr/local/lib/python3.12/dist-packages/fastprogress/fastprogress.py:93: UserWarning: Your generator is empty.
  warn("Your generator is empty.")
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(

epoch,train_loss,valid_loss,error_rate,time
0,nan,1.617345,0.500000,00:01
1,nan,1.617345,0.500000,00:01


/usr/local/lib/python3.12/dist-packages/fastprogress/fastprogress.py:93: UserWarning: Your generator is empty.
  warn("Your generator is empty.")
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284f930>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(zmq.PUSH) at 0x7ea44284ee40>
  self.pid = os.fork()
/usr/lib/python3.12/multiprocessing/popen_fork.py:66: ResourceWarning: Unclosed socket <zmq.Socket(

Model trained and exported as export.pkl


## 5. Interactive Classifier App
This section creates a graphical interface where you can upload an image and get an instant classification prediction.

In [128]:
learn_inf = load_learner('export.pkl')
btn_upload = widgets.FileUpload()
out_pl = widgets.Output()
lbl_pred = widgets.Label()
btn_run = widgets.Button(description='Classify', button_style='primary')

def on_click_classify(change):
    img = PILImage.create(next(iter(btn_upload.value.values()))['content'])
    out_pl.clear_output()
    with out_pl: display(img.to_thumb(256,256))
    pred,pred_idx,probs = learn_inf.predict(img)
    lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

btn_run.on_click(on_click_classify)
widgets.VBox([widgets.Label('Select your bear!'), btn_upload, btn_run, out_pl, lbl_pred])

/usr/local/lib/python3.12/dist-packages/fastai/learner.py:455: UserWarning: load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.
If you only need to load model weights and optimizer state, use the safe `Learner.load` instead.
  warn("load_learner` uses Python's insecure pickle module, which can execute malicious arbitrary code when loading. Only load files you trust.\nIf you only need to load model weights and optimizer state, use the safe `Learner.load` instead.")


### 6. Fix for 'Invalid Notebook' Metadata Error
Run the cell below to clean the notebook metadata. This ensures that the `metadata.widgets` key is either properly formatted or removed, preventing rendering errors on GitHub.

In [130]:
import nbformat
from google.colab import files

# This script cleans the metadata that often causes GitHub rendering issues
def clean_notebook_widgets(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        nb = nbformat.read(f, as_version=4)

    # Check if widgets metadata exists and ensure 'state' is present or remove it
    if 'widgets' in nb.metadata:
        if 'state' not in nb.metadata['widgets']:
            print("Cleaning incomplete widget metadata...")
            del nb.metadata['widgets']

    # Save the cleaned version
    cleaned_path = file_path.replace('.ipynb', '_cleaned.ipynb')
    with open(cleaned_path, 'w', encoding='utf-8') as f:
        nbformat.write(nb, f)
    print(f"Cleaned notebook saved as: {cleaned_path}")

# Note: To use this, you would need to save the notebook as a file first.
# For now, this is a utility you can run before exporting to GitHub.
print("Ready to clean metadata if a file path is provided.")

Ready to clean metadata if a file path is provided.
